# Loan Approval Machine-Learning Project

## Project Purpose

The purpose of this project is to develop a responsible, explainable, and maintainable machine-learning model that estimates an applicant's loan default risk. The model will produce a score or probability that supports loan approval decisions while keeping lending policy, compliance review, and appropriate human oversight separate from the model prediction.

The project will evaluate interpretable baseline models and advanced classifiers, with particular attention to probability calibration, predictive performance, fairness, stability, and explainability. The resulting model is intended to support approval, manual-review, or decline workflows rather than make ungoverned lending decisions.

Project development and evaluation activities are tracked in `PROJECT_PLAN.md`, and the candidate model recommendations are documented in `MODEL_RECOMENDED.md`.

## Software Installation

Use Python 3.11.9 or newer in a project-specific virtual environment. Install the required packages with:

```bash
python -m pip install --upgrade pip
python -m pip install jupyterlab numpy pandas matplotlib seaborn scikit-learn
```

### Package purposes

- `jupyterlab`: runs and edits the project notebook.
- `numpy`: provides numerical array and mathematical operations.
- `pandas`: loads, validates, transforms, and analyzes tabular loan data.
- `matplotlib`: creates foundational charts and diagnostic plots.
- `seaborn`: creates statistical data visualizations.
- `scikit-learn`: provides preprocessing, classifiers, probability calibration, metrics, and model-validation tools.

The standard library and these packages are sufficient for the initial logistic-regression, credit-scorecard preparation, decision-tree, random-forest, and histogram gradient-boosting candidates. Add other libraries only when a documented requirement cannot be met by the existing environment.

In [174]:
%pip install --upgrade pip
%pip install jupyterlab numpy pandas matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [175]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [176]:
# Data configuration
DATA_PATH = Path("data") / "loan_applications.csv"
ID_COLUMN = "applicant_id"
TARGET_COLUMN = "approved"
POSITIVE_CLASS = "Approved"

NUMERIC_FEATURES = (
    "fico_score",
    "annual_income",
    "loan_amount",
    "loan_term_months",
    "years_employed",
    "savings_balance",
)
CATEGORICAL_FEATURES = ("employment_status",)
INPUT_COLUMNS = [
    "fico_score",
    "annual_income",
    "loan_amount",
    "loan_term_months",
    "employment_status",
    "years_employed",
    "savings_balance",
]

# Training and validation configuration
RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
CROSS_VALIDATION_FOLDS = 5
CALIBRATION_METHOD = "sigmoid"
MINIMUM_TRAINING_ROWS = 100
MINIMUM_CLASS_COUNT = 20
MAXIMUM_MISSING_RATE = 0.20

# Candidate model configuration
LOGISTIC_REGRESSION_PARAMS = {
    "class_weight": "balanced",
    "max_iter": 1_000,
    "random_state": RANDOM_STATE,
}
DECISION_TREE_PARAMS = {
    "class_weight": "balanced",
    "max_depth": 5,
    "min_samples_leaf": 10,
    "random_state": RANDOM_STATE,
}
RANDOM_FOREST_PARAMS = {
    "class_weight": "balanced",
    "min_samples_leaf": 5,
    "n_estimators": 300,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}
GRADIENT_BOOSTING_PARAMS = {
    "learning_rate": 0.05,
    "max_depth": 3,
    "n_estimators": 200,
    "random_state": RANDOM_STATE,
}
MLP_CLASSIFIER_PARAMS = {
    "early_stopping": True,
    "hidden_layer_sizes": (64, 32),
    "learning_rate_init": 0.001,
    "max_iter": 1_000,
    "random_state": RANDOM_STATE,
    "validation_fraction": VALIDATION_SIZE,
}
SVC_PARAMS = {
    "class_weight": "balanced",
    "gamma": "scale",
    "kernel": "rbf",
    "random_state": RANDOM_STATE,
}
GAUSSIAN_NB_PARAMS = {
    "var_smoothing": 1e-9,
}

In [177]:
def validate_split_data(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    train_target: pd.Series,
    test_target: pd.Series,
) -> None:
    """Validate model training and test inputs."""
    if train_features.empty or test_features.empty:
        raise ValueError("Training and test features cannot be empty.")
    if len(train_features) != len(train_target):
        raise ValueError("Training features and target must have equal lengths.")
    if len(test_features) != len(test_target):
        raise ValueError("Test features and target must have equal lengths.")

    missing_columns = set(INPUT_COLUMNS) - set(train_features.columns)
    missing_columns |= set(INPUT_COLUMNS) - set(test_features.columns)
    if missing_columns:
        missing_names = ", ".join(sorted(missing_columns))
        raise ValueError(f"Model input columns are missing: {missing_names}.")


def create_preprocessor(
    *,
    scale_numeric: bool,
    dense_output: bool,
) -> ColumnTransformer:
    """Create shared numeric and categorical preprocessing."""
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_preprocessor = Pipeline(steps=numeric_steps)
    categorical_preprocessor = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=not dense_output,
                ),
            ),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_preprocessor, list(NUMERIC_FEATURES)),
            ("categorical", categorical_preprocessor, list(CATEGORICAL_FEATURES)),
        ]
    )


def evaluate_model(
    model: Pipeline,
    test_features: pd.DataFrame,
    test_target: pd.Series,
) -> tuple[float, float, np.ndarray, str]:
    """Calculate shared binary-classification test metrics."""
    test_inputs = test_features.loc[:, INPUT_COLUMNS]
    test_predictions = model.predict(test_inputs)
    test_probabilities = model.predict_proba(test_inputs)[:, 1]
    return (
        float(roc_auc_score(test_target, test_probabilities)),
        float(accuracy_score(test_target, test_predictions)),
        confusion_matrix(test_target, test_predictions),
        classification_report(test_target, test_predictions, zero_division=0),
    )

In [178]:
class logistic_regression:
    """Placeholder for the logistic-regression model implementation."""

    def get_model(
        self,
        train_features: pd.DataFrame,
        test_features: pd.DataFrame,
        train_target: pd.Series,
        test_target: pd.Series,
    ) -> tuple[Pipeline, float]:
        """Train a logistic-regression pipeline and return its test accuracy."""
        validate_split_data(
            train_features,
            test_features,
            train_target,
            test_target,
        )

        preprocessor = create_preprocessor(
            scale_numeric=True,
            dense_output=False,
        )
        model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "classifier",
                    LogisticRegression(**LOGISTIC_REGRESSION_PARAMS),
                ),
            ]
        )

        model.fit(train_features.loc[:, INPUT_COLUMNS], train_target)
        _, model_accuracy, _, _ = evaluate_model(
            model,
            test_features,
            test_target,
        )

        return model, model_accuracy

In [179]:
class credit_scorecard:
    """Placeholder for the credit-scorecard model implementation."""

    def get_model(
        self,
        train_features: pd.DataFrame,
        test_features: pd.DataFrame,
        train_target: pd.Series,
        test_target: pd.Series,
    ) -> tuple[Pipeline, float]:
        """Train a credit-scorecard pipeline and return its test accuracy."""
        validate_split_data(
            train_features,
            test_features,
            train_target,
            test_target,
        )

        preprocessor = create_preprocessor(
            scale_numeric=True,
            dense_output=False,
        )
        model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "classifier",
                    LogisticRegression(**LOGISTIC_REGRESSION_PARAMS),
                ),
            ]
        )

        model.fit(train_features.loc[:, INPUT_COLUMNS], train_target)
        (
            model_roc_auc,
            model_accuracy,
            model_confusion_matrix,
            model_classification_report,
        ) = evaluate_model(
            model,
            test_features,
            test_target,
        )

        print(f"Accuracy score: {model_accuracy:.4f}")
        print(f"ROC-AUC score: {model_roc_auc:.4f}")
        print("Confusion matrix:")
        print(model_confusion_matrix)
        print("Classification report:")
        print(model_classification_report)

        return model, model_accuracy

In [180]:
class decision_tree:
    """Placeholder for the decision-tree model implementation."""

    def get_model(
        self,
        train_features: pd.DataFrame,
        test_features: pd.DataFrame,
        train_target: pd.Series,
        test_target: pd.Series,
    ) -> tuple[Pipeline, np.ndarray, float]:
        """Train a decision-tree pipeline and return test metrics."""
        validate_split_data(
            train_features,
            test_features,
            train_target,
            test_target,
        )

        preprocessor = create_preprocessor(
            scale_numeric=False,
            dense_output=False,
        )
        model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "classifier",
                    DecisionTreeClassifier(**DECISION_TREE_PARAMS),
                ),
            ]
        )

        model.fit(train_features.loc[:, INPUT_COLUMNS], train_target)
        (
            _,
            model_accuracy,
            model_confusion_matrix,
            _,
        ) = evaluate_model(model, test_features, test_target)

        return model, model_confusion_matrix, model_accuracy

In [181]:
class random_forest:
    """Placeholder for the random-forest model implementation."""

    def get_model(
        self,
        train_features: pd.DataFrame,
        test_features: pd.DataFrame,
        train_target: pd.Series,
        test_target: pd.Series,
    ) -> tuple[Pipeline, np.ndarray, float, str]:
        """Train a random-forest pipeline and return test metrics."""
        validate_split_data(
            train_features,
            test_features,
            train_target,
            test_target,
        )

        preprocessor = create_preprocessor(
            scale_numeric=False,
            dense_output=False,
        )
        model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "classifier",
                    RandomForestClassifier(**RANDOM_FOREST_PARAMS),
                ),
            ]
        )

        model.fit(train_features.loc[:, INPUT_COLUMNS], train_target)
        (
            _,
            model_accuracy,
            model_confusion_matrix,
            model_classification_report,
        ) = evaluate_model(
            model,
            test_features,
            test_target,
        )

        return (
            model,
            model_confusion_matrix,
            model_accuracy,
            model_classification_report,
        )

In [182]:
class gradient_boosted_trees:
    """Placeholder for the gradient-boosted-trees implementation."""

    def get_model(
        self,
        train_features: pd.DataFrame,
        test_features: pd.DataFrame,
        train_target: pd.Series,
        test_target: pd.Series,
    ) -> tuple[Pipeline, float, float, np.ndarray, str]:
        """Train a gradient-boosting pipeline and return test metrics."""
        validate_split_data(
            train_features,
            test_features,
            train_target,
            test_target,
        )

        preprocessor = create_preprocessor(
            scale_numeric=False,
            dense_output=True,
        )
        model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "classifier",
                    GradientBoostingClassifier(**GRADIENT_BOOSTING_PARAMS),
                ),
            ]
        )

        model.fit(train_features.loc[:, INPUT_COLUMNS], train_target)
        (
            model_roc_auc,
            model_accuracy,
            model_confusion_matrix,
            model_classification_report,
        ) = evaluate_model(
            model,
            test_features,
            test_target,
        )

        return (
            model,
            model_roc_auc,
            model_accuracy,
            model_confusion_matrix,
            model_classification_report,
        )

In [183]:
class neural_network:
    """Placeholder for the neural-network model implementation."""

    def get_model(
        self,
        train_features: pd.DataFrame,
        test_features: pd.DataFrame,
        train_target: pd.Series,
        test_target: pd.Series,
    ) -> tuple[Pipeline, float, float, np.ndarray, str]:
        """Train an MLP classifier pipeline and return test metrics."""
        validate_split_data(
            train_features,
            test_features,
            train_target,
            test_target,
        )

        preprocessor = create_preprocessor(
            scale_numeric=True,
            dense_output=True,
        )
        model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("classifier", MLPClassifier(**MLP_CLASSIFIER_PARAMS)),
            ]
        )

        model.fit(train_features.loc[:, INPUT_COLUMNS], train_target)
        (
            model_roc_auc,
            model_accuracy,
            model_confusion_matrix,
            model_classification_report,
        ) = evaluate_model(
            model,
            test_features,
            test_target,
        )

        return (
            model,
            model_roc_auc,
            model_accuracy,
            model_confusion_matrix,
            model_classification_report,
        )

In [184]:
class support_vector_machine:
    """Placeholder for the support-vector-machine implementation."""

    def get_model(
        self,
        train_features: pd.DataFrame,
        test_features: pd.DataFrame,
        train_target: pd.Series,
        test_target: pd.Series,
    ) -> tuple[Pipeline, float, float, np.ndarray, str]:
        """Train an SVC pipeline and return test metrics."""
        validate_split_data(
            train_features,
            test_features,
            train_target,
            test_target,
        )

        preprocessor = create_preprocessor(
            scale_numeric=True,
            dense_output=True,
        )
        model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "classifier",
                    CalibratedClassifierCV(
                        estimator=SVC(**SVC_PARAMS),
                        method=CALIBRATION_METHOD,
                        cv=CROSS_VALIDATION_FOLDS,
                        ensemble=False,
                    ),
                ),
            ]
        )

        model.fit(train_features.loc[:, INPUT_COLUMNS], train_target)
        (
            model_roc_auc,
            model_accuracy,
            model_confusion_matrix,
            model_classification_report,
        ) = evaluate_model(
            model,
            test_features,
            test_target,
        )

        return (
            model,
            model_roc_auc,
            model_accuracy,
            model_confusion_matrix,
            model_classification_report,
        )

In [185]:
class naive_bayes:
    """Placeholder for the naive-Bayes model implementation."""

    def get_model(
        self,
        train_features: pd.DataFrame,
        test_features: pd.DataFrame,
        train_target: pd.Series,
        test_target: pd.Series,
    ) -> tuple[Pipeline, float, float, np.ndarray, str]:
        """Train a Gaussian naive-Bayes pipeline and return test metrics."""
        validate_split_data(
            train_features,
            test_features,
            train_target,
            test_target,
        )

        preprocessor = create_preprocessor(
            scale_numeric=True,
            dense_output=True,
        )
        model = Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                ("classifier", GaussianNB(**GAUSSIAN_NB_PARAMS)),
            ]
        )

        model.fit(train_features.loc[:, INPUT_COLUMNS], train_target)
        (
            model_roc_auc,
            model_accuracy,
            model_confusion_matrix,
            model_classification_report,
        ) = evaluate_model(
            model,
            test_features,
            test_target,
        )

        return (
            model,
            model_roc_auc,
            model_accuracy,
            model_confusion_matrix,
            model_classification_report,
        )

## Step 2: Prepare the Data

Prepare a reliable and reproducible dataset for model training, validation, and testing. All transformations must use only information available at the time of the loan application.

### Tasks

- [DONE] Validate required fields, data types, formats, ranges, and allowed values.
- [DONE] Identify duplicate applications and inconsistent records.
- [DONE] Measure missing values and define an explicit treatment for each feature.
- [DONE] Detect target leakage and features that may act as inappropriate proxies.
- [DONE] Exclude `applicant_id` from model features while retaining it for traceability.
- [TODO] Encode the `approved` target consistently and document the positive class.
- [TODO] Examine target-class imbalance and document any treatment applied.
- [TODO] Create reproducible training, validation, and test datasets.
- [TODO] Fit preprocessing operations on training data only to prevent leakage.
- [TODO] Document the final feature set and preprocessing decisions.

### Completion criteria

Step 2 is complete when the prepared datasets pass the documented quality checks, contain no known target leakage, and can be reproduced from the original source data.

In [186]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Loan application data was not found: {DATA_PATH}")

try:
    loan_applications = pd.read_csv(DATA_PATH)
except pd.errors.EmptyDataError as error:
    raise ValueError(f"Loan application data is empty: {DATA_PATH}") from error
except pd.errors.ParserError as error:
    raise ValueError(f"Loan application data could not be parsed: {DATA_PATH}") from error

print("Schema before cleaning:")
print(loan_applications.dtypes)

currency_columns = ["annual_income", "loan_amount", "savings_balance"]
loan_applications[currency_columns] = loan_applications[currency_columns].apply(
    lambda column: pd.to_numeric(
        column.astype("string").str.replace(r"[$,']", "", regex=True),
        errors="coerce",
    )
)

loan_applications = loan_applications.loc[
    ~loan_applications["annual_income"].lt(0)
].copy()

loan_applications["fico_score"] = (
    pd.to_numeric(loan_applications["fico_score"], errors="coerce")
    .fillna(0)
    .astype("int64")
)

loan_applications = loan_applications.drop_duplicates(
    subset=ID_COLUMN,
    keep="first",
).copy()

print("\nSchema after cleaning:")
print(loan_applications.dtypes)

loan_applications.head()

Schema before cleaning:
applicant_id             str
fico_score           float64
annual_income            str
loan_amount              str
loan_term_months       int64
employment_status        str
years_employed         int64
savings_balance          str
approved                 str
dtype: object

Schema after cleaning:
applicant_id           str
fico_score           int64
annual_income        Int64
loan_amount          Int64
loan_term_months     int64
employment_status      str
years_employed       int64
savings_balance      Int64
approved               str
dtype: object


,applicant_id,fico_score,annual_income,loan_amount,loan_term_months,employment_status,years_employed,savings_balance,approved
0,BCC-1219,0,69100,12000,48,Self-employed,20,19200,Approved
1,BCC-1066,646,64900,19500,24,Salaried,12,21100,Approved
2,BCC-1009,610,50500,9200,36,Self-employed,12,7300,Approved
3,BCC-1170,624,20000,10800,12,Salaried,11,8700,Approved
4,BCC-1015,610,41000,10400,36,Salaried,17,3800,Approved


In [187]:
required_columns = {ID_COLUMN, TARGET_COLUMN, *INPUT_COLUMNS}
missing_columns = required_columns - set(loan_applications.columns)
if missing_columns:
    missing_names = ", ".join(sorted(missing_columns))
    raise ValueError(f"Required columns are missing: {missing_names}.")

row_count = len(loan_applications)
duplicate_applicant_count = int(loan_applications[ID_COLUMN].duplicated().sum())
missing_rates = loan_applications.loc[:, INPUT_COLUMNS].isna().mean()
target_class_counts = loan_applications[TARGET_COLUMN].value_counts(dropna=False)
invalid_numeric_counts = {
    "annual_income": int((loan_applications["annual_income"] < 0).sum()),
    "loan_amount": int((loan_applications["loan_amount"] <= 0).sum()),
    "loan_term_months": int((loan_applications["loan_term_months"] <= 0).sum()),
    "years_employed": int((loan_applications["years_employed"] < 0).sum()),
    "savings_balance": int((loan_applications["savings_balance"] < 0).sum()),
}

quality_violations = []
if row_count < MINIMUM_TRAINING_ROWS:
    quality_violations.append(
        f"Only {row_count} rows are available; at least {MINIMUM_TRAINING_ROWS} are required."
    )
if duplicate_applicant_count:
    quality_violations.append(
        f"Found {duplicate_applicant_count} duplicate applicant IDs."
    )
if missing_rates.max() > MAXIMUM_MISSING_RATE:
    quality_violations.append(
        f"At least one feature exceeds the {MAXIMUM_MISSING_RATE:.0%} missing-value limit."
    )
if loan_applications[TARGET_COLUMN].isna().any():
    quality_violations.append("The target column contains missing values.")
if target_class_counts.min() < MINIMUM_CLASS_COUNT:
    quality_violations.append(
        f"At least one target class has fewer than {MINIMUM_CLASS_COUNT} rows."
    )
if any(invalid_numeric_counts.values()):
    quality_violations.append("One or more numeric features contain invalid values.")

data_quality_report = {
    "row_count": row_count,
    "duplicate_applicant_count": duplicate_applicant_count,
    "missing_rates": missing_rates.to_dict(),
    "target_class_counts": target_class_counts.to_dict(),
    "invalid_numeric_counts": invalid_numeric_counts,
    "passed": not quality_violations,
}

if quality_violations:
    raise ValueError("Data quality checks failed: " + " ".join(quality_violations))

data_quality_report

{'row_count': 218,
 'duplicate_applicant_count': 0,
 'missing_rates': {'fico_score': 0.0,
  'annual_income': 0.0,
  'loan_amount': 0.0,
  'loan_term_months': 0.0,
  'employment_status': 0.0,
  'years_employed': 0.0,
  'savings_balance': 0.0},
 'target_class_counts': {'Approved': 131, 'Rejected': 87},
 'invalid_numeric_counts': {'annual_income': 0,
  'loan_amount': 0,
  'loan_term_months': 0,
  'years_employed': 0,
  'savings_balance': 0},
 'passed': True}

In [188]:
input_features = loan_applications.loc[:, INPUT_COLUMNS].copy()
target = loan_applications[TARGET_COLUMN].eq(POSITIVE_CLASS).astype("int8")

train_validation_features, test_features, train_validation_target, test_target = (
    train_test_split(
        input_features,
        target,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=target,
    )
)

validation_fraction_of_remaining = VALIDATION_SIZE / (1 - TEST_SIZE)
train_features, validation_features, train_target, validation_target = (
    train_test_split(
        train_validation_features,
        train_validation_target,
        test_size=validation_fraction_of_remaining,
        random_state=RANDOM_STATE,
        stratify=train_validation_target,
    )
)

dataset_split_summary = pd.DataFrame(
    {
        "rows": [len(train_features), len(validation_features), len(test_features)],
        "positive_rate": [
            train_target.mean(),
            validation_target.mean(),
            test_target.mean(),
        ],
    },
    index=["train", "validation", "test"],
)

dataset_split_summary

,rows,positive_rate
train,130,0.600000
validation,44,0.613636
test,44,0.590909


In [189]:
logistic_regression_instance = logistic_regression()
trained_logistic_model, logistic_accuracy = (
    logistic_regression_instance.get_model(
        train_features=train_features,
        test_features=test_features,
        train_target=train_target,
        test_target=test_target,
    )
)
print(f"Logistic regression accuracy score: {logistic_accuracy:.4f}")

sample_customers = pd.DataFrame(
    [
        {
            "fico_score": 820,
            "annual_income": 120_000,
            "loan_amount": 5_000,
            "loan_term_months": 12,
            "employment_status": "Salaried",
            "years_employed": 15,
            "savings_balance": 40_000,
        },
        {
            "fico_score": 450,
            "annual_income": 15_000,
            "loan_amount": 45_000,
            "loan_term_months": 60,
            "employment_status": "Self-employed",
            "years_employed": 0,
            "savings_balance": 0,
        },
    ],
    index=["positive_case", "negative_case"],
)

sample_predictions = trained_logistic_model.predict(sample_customers)
sample_probabilities = trained_logistic_model.predict_proba(sample_customers)[:, 1]
prediction_results = pd.DataFrame(
    {
        "prediction": np.where(
            sample_predictions == 1,
            POSITIVE_CLASS,
            "Not Approved",
        ),
        "approval_probability": sample_probabilities,
    },
    index=sample_customers.index,
)

print("\nSample customer predictions:")
print(prediction_results)

Logistic regression accuracy score: 0.8409

Sample customer predictions:
                 prediction  approval_probability
positive_case      Approved              0.999525
negative_case  Not Approved              0.001111


In [190]:
credit_scorecard_instance = credit_scorecard()
trained_scorecard_model, scorecard_accuracy = (
    credit_scorecard_instance.get_model(
        train_features=train_features,
        test_features=test_features,
        train_target=train_target,
        test_target=test_target,
    )
)
print(f"Credit scorecard accuracy score: {scorecard_accuracy:.4f}")

scorecard_predictions = trained_scorecard_model.predict(sample_customers)
scorecard_probabilities = trained_scorecard_model.predict_proba(
    sample_customers
)[:, 1]

minimum_score_points = 300
maximum_score_points = 850
scorecard_points = np.rint(
    minimum_score_points
    + scorecard_probabilities * (maximum_score_points - minimum_score_points)
).astype("int64")

scorecard_results = pd.DataFrame(
    {
        "prediction": np.where(
            scorecard_predictions == 1,
            POSITIVE_CLASS,
            "Not Approved",
        ),
        "approval_probability": scorecard_probabilities,
        "score_points": scorecard_points,
    },
    index=sample_customers.index,
)

print("\nCredit scorecard sample predictions and points:")
print(scorecard_results)

Accuracy score: 0.8409
ROC-AUC score: 0.9081
Confusion matrix:
[[15  3]
 [ 4 22]]
Classification report:
              precision    recall  f1-score   support

           0       0.79      0.83      0.81        18
           1       0.88      0.85      0.86        26

    accuracy                           0.84        44
   macro avg       0.83      0.84      0.84        44
weighted avg       0.84      0.84      0.84        44

Credit scorecard accuracy score: 0.8409

Credit scorecard sample predictions and points:
                 prediction  approval_probability  score_points
positive_case      Approved              0.999525           850
negative_case  Not Approved              0.001111           301


In [191]:
decision_tree_instance = decision_tree()
(
    trained_decision_tree,
    decision_tree_confusion_matrix,
    decision_tree_accuracy,
) = decision_tree_instance.get_model(
        train_features=train_features,
        test_features=test_features,
        train_target=train_target,
        test_target=test_target,
)

print("Decision tree confusion matrix:")
print(decision_tree_confusion_matrix)
print(f"Decision tree accuracy score: {decision_tree_accuracy:.4f}")

decision_tree_predictions = trained_decision_tree.predict(sample_customers)
decision_tree_probabilities = trained_decision_tree.predict_proba(
    sample_customers
)[:, 1]
decision_tree_results = pd.DataFrame(
    {
        "prediction": np.where(
            decision_tree_predictions == 1,
            POSITIVE_CLASS,
            "Not Approved",
        ),
        "approval_probability": decision_tree_probabilities,
    },
    index=sample_customers.index,
)

print("\nDecision tree sample predictions and probabilities:")
print(decision_tree_results)

Decision tree confusion matrix:
[[13  5]
 [ 9 17]]
Decision tree accuracy score: 0.6818

Decision tree sample predictions and probabilities:
                 prediction  approval_probability
positive_case      Approved              1.000000
negative_case  Not Approved              0.037736


In [192]:
random_forest_instance = random_forest()
(
    trained_random_forest,
    random_forest_confusion_matrix,
    random_forest_accuracy,
    random_forest_classification_report,
) = random_forest_instance.get_model(
    train_features=train_features,
    test_features=test_features,
    train_target=train_target,
    test_target=test_target,
)

print("Random forest confusion matrix:")
print(random_forest_confusion_matrix)
print(f"Random forest accuracy score: {random_forest_accuracy:.4f}")
print("Random forest classification report:")
print(random_forest_classification_report)

random_forest_predictions = trained_random_forest.predict(sample_customers)
random_forest_probabilities = trained_random_forest.predict_proba(
    sample_customers
)[:, 1]
random_forest_results = pd.DataFrame(
    {
        "prediction": np.where(
            random_forest_predictions == 1,
            POSITIVE_CLASS,
            "Not Approved",
        ),
        "approval_probability": random_forest_probabilities,
    },
    index=sample_customers.index,
)

print("\nRandom forest sample predictions and probabilities:")
print(random_forest_results)

Random forest confusion matrix:
[[13  5]
 [ 3 23]]
Random forest accuracy score: 0.8182
Random forest classification report:
              precision    recall  f1-score   support

           0       0.81      0.72      0.76        18
           1       0.82      0.88      0.85        26

    accuracy                           0.82        44
   macro avg       0.82      0.80      0.81        44
weighted avg       0.82      0.82      0.82        44


Random forest sample predictions and probabilities:
                 prediction  approval_probability
positive_case      Approved              0.906983
negative_case  Not Approved              0.136291


In [193]:
gradient_boosted_trees_instance = gradient_boosted_trees()

(
    trained_gradient_boosted_trees,
    gradient_boosted_trees_roc_auc,
    gradient_boosted_trees_accuracy,
    gradient_boosted_trees_confusion_matrix,
    gradient_boosted_trees_classification_report,
) = gradient_boosted_trees_instance.get_model(
    train_features=train_features,
    test_features=test_features,
    train_target=train_target,
    test_target=test_target,
)

print(
    "Gradient boosted trees ROC-AUC score: "
    f"{gradient_boosted_trees_roc_auc:.4f}"
)
print(
    "Gradient boosted trees accuracy score: "
    f"{gradient_boosted_trees_accuracy:.4f}"
)
print("Gradient boosted trees confusion matrix:")
print(gradient_boosted_trees_confusion_matrix)
print("Gradient boosted trees classification report:")
print(gradient_boosted_trees_classification_report)

gradient_boosted_trees_predictions = trained_gradient_boosted_trees.predict(
    sample_customers
)
gradient_boosted_trees_probabilities = (
    trained_gradient_boosted_trees.predict_proba(sample_customers)[:, 1]
)
gradient_boosted_trees_results = pd.DataFrame(
    {
        "prediction": np.where(
            gradient_boosted_trees_predictions == 1,
            POSITIVE_CLASS,
            "Not Approved",
        ),
        "approval_probability": gradient_boosted_trees_probabilities,
    },
    index=sample_customers.index,
)

print("\nGradient boosted trees sample predictions and probabilities:")
print(gradient_boosted_trees_results)

Gradient boosted trees ROC-AUC score: 0.8654
Gradient boosted trees accuracy score: 0.6818
Gradient boosted trees confusion matrix:
[[ 6 12]
 [ 2 24]]
Gradient boosted trees classification report:
              precision    recall  f1-score   support

           0       0.75      0.33      0.46        18
           1       0.67      0.92      0.77        26

    accuracy                           0.68        44
   macro avg       0.71      0.63      0.62        44
weighted avg       0.70      0.68      0.65        44


Gradient boosted trees sample predictions and probabilities:
                 prediction  approval_probability
positive_case      Approved              0.997875
negative_case  Not Approved              0.000192


In [194]:
neural_network_instance = neural_network()
(
    trained_neural_network,
    neural_network_roc_auc,
    neural_network_accuracy,
    neural_network_confusion_matrix,
    neural_network_classification_report,
) = neural_network_instance.get_model(
    train_features=train_features,
    test_features=test_features,
    train_target=train_target,
    test_target=test_target,
)

print(f"Neural network ROC-AUC score: {neural_network_roc_auc:.4f}")
print(f"Neural network accuracy score: {neural_network_accuracy:.4f}")
print("Neural network confusion matrix:")
print(neural_network_confusion_matrix)
print("Neural network classification report:")
print(neural_network_classification_report)

neural_network_predictions = trained_neural_network.predict(sample_customers)
neural_network_probabilities = trained_neural_network.predict_proba(
    sample_customers
)[:, 1]
neural_network_results = pd.DataFrame(
    {
        "prediction": np.where(
            neural_network_predictions == 1,
            POSITIVE_CLASS,
            "Not Approved",
        ),
        "approval_probability": neural_network_probabilities,
    },
    index=sample_customers.index,
)

print("\nNeural network sample predictions and probabilities:")
print(neural_network_results)

Neural network ROC-AUC score: 0.9038
Neural network accuracy score: 0.6818
Neural network confusion matrix:
[[ 4 14]
 [ 0 26]]
Neural network classification report:
              precision    recall  f1-score   support

           0       1.00      0.22      0.36        18
           1       0.65      1.00      0.79        26

    accuracy                           0.68        44
   macro avg       0.82      0.61      0.58        44
weighted avg       0.79      0.68      0.61        44


Neural network sample predictions and probabilities:
                 prediction  approval_probability
positive_case      Approved              0.706094
negative_case  Not Approved              0.441309


In [195]:
support_vector_machine_instance = support_vector_machine()
(
    trained_support_vector_machine,
    support_vector_machine_roc_auc,
    support_vector_machine_accuracy,
    support_vector_machine_confusion_matrix,
    support_vector_machine_classification_report,
) = support_vector_machine_instance.get_model(
    train_features=train_features,
    test_features=test_features,
    train_target=train_target,
    test_target=test_target,
)

print(
    "Support vector machine ROC-AUC score: "
    f"{support_vector_machine_roc_auc:.4f}"
)
print(
    "Support vector machine accuracy score: "
    f"{support_vector_machine_accuracy:.4f}"
)
print("Support vector machine confusion matrix:")
print(support_vector_machine_confusion_matrix)
print("Support vector machine classification report:")
print(support_vector_machine_classification_report)

support_vector_machine_predictions = trained_support_vector_machine.predict(
    sample_customers
)
support_vector_machine_probabilities = (
    trained_support_vector_machine.predict_proba(sample_customers)[:, 1]
)
support_vector_machine_results = pd.DataFrame(
    {
        "prediction": np.where(
            support_vector_machine_predictions == 1,
            POSITIVE_CLASS,
            "Not Approved",
        ),
        "approval_probability": support_vector_machine_probabilities,
    },
    index=sample_customers.index,
)

print("\nSupport vector machine sample predictions and probabilities:")
print(support_vector_machine_results)

Support vector machine ROC-AUC score: 0.9316
Support vector machine accuracy score: 0.7955
Support vector machine confusion matrix:
[[11  7]
 [ 2 24]]
Support vector machine classification report:
              precision    recall  f1-score   support

           0       0.85      0.61      0.71        18
           1       0.77      0.92      0.84        26

    accuracy                           0.80        44
   macro avg       0.81      0.77      0.78        44
weighted avg       0.80      0.80      0.79        44


Support vector machine sample predictions and probabilities:
                 prediction  approval_probability
positive_case      Approved              0.936306
negative_case  Not Approved              0.079786


In [196]:
naive_bayes_instance = naive_bayes()
(
    trained_naive_bayes,
    naive_bayes_roc_auc,
    naive_bayes_accuracy,
    naive_bayes_confusion_matrix,
    naive_bayes_classification_report,
) = naive_bayes_instance.get_model(
    train_features=train_features,
    test_features=test_features,
    train_target=train_target,
    test_target=test_target,
)

print(f"Naive Bayes ROC-AUC score: {naive_bayes_roc_auc:.4f}")
print(f"Naive Bayes accuracy score: {naive_bayes_accuracy:.4f}")
print("Naive Bayes confusion matrix:")
print(naive_bayes_confusion_matrix)
print("Naive Bayes classification report:")
print(naive_bayes_classification_report)

naive_bayes_predictions = trained_naive_bayes.predict(sample_customers)
naive_bayes_probabilities = trained_naive_bayes.predict_proba(
    sample_customers
)[:, 1]
naive_bayes_results = pd.DataFrame(
    {
        "prediction": np.where(
            naive_bayes_predictions == 1,
            POSITIVE_CLASS,
            "Not Approved",
        ),
        "approval_probability": naive_bayes_probabilities,
    },
    index=sample_customers.index,
)

print("\nNaive Bayes sample predictions and probabilities:")
print(naive_bayes_results)

Naive Bayes ROC-AUC score: 0.8846
Naive Bayes accuracy score: 0.5000
Naive Bayes confusion matrix:
[[18  0]
 [22  4]]
Naive Bayes classification report:
              precision    recall  f1-score   support

           0       0.45      1.00      0.62        18
           1       1.00      0.15      0.27        26

    accuracy                           0.50        44
   macro avg       0.72      0.58      0.44        44
weighted avg       0.78      0.50      0.41        44


Naive Bayes sample predictions and probabilities:
                 prediction  approval_probability
positive_case      Approved              0.947758
negative_case  Not Approved              0.033403
